# 04_SQL

En esta sección se trasladan los datasets limpios a una base de datos MySQL y se ejecutan consultas SQL orientadas a negocio para analizar:

- evolución del mercado automovilístico
- tamaño y renovación del parque
- estructura del mercado por tipo de vehículo
- transición energética
- concentración geográfica
- adopción de vehículos de bajas emisiones

In [ ]:
#CREANDO LOS ARCHIVOS PARA VERLOS EN MYSQL
from pathlib import Path
import pandas as pd

In [ ]:
PROCESSED_DIR = Path("/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Automocion/data/processed")
SQL_DIR = Path("/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Automocion/sql")

In [44]:
matr_tipo = pd.read_csv(PROCESSED_DIR / "matriculaciones_tipo_limpio.csv")
parque = pd.read_csv(PROCESSED_DIR / "parque_vehiculos_limpio.csv")
matr_prov = pd.read_csv(PROCESSED_DIR / "matriculaciones_provincia.csv")
distintivo = pd.read_csv(PROCESSED_DIR / "distintivo_ambiental.csv")
combustible = pd.read_csv(PROCESSED_DIR / "combustible.csv")

In [46]:
#MATRICULACIONES

lines = []
lines.append("USE automocion;")
lines.append("DROP TABLE IF EXISTS matriculaciones;")
lines.append("""
CREATE TABLE matriculaciones (
    anio INT,
    camiones_furgonetas INT,
    autobuses INT,
    turismos INT,
    motocicletas INT,
    tractores_industriales INT,
    otros_vehiculos INT,
    total INT
);
""")

for _, row in matr_tipo.iterrows():
    lines.append(
        f"INSERT INTO matriculaciones VALUES "
        f"({int(row['anio'])}, {int(row['camiones_furgonetas'])}, {int(row['autobuses'])}, "
        f"{int(row['turismos'])}, {int(row['motocicletas'])}, "
        f"{int(row['tractores_industriales'])}, {int(row['otros_vehiculos'])}, "
        f"{int(row['total'])});"
    )

with open(SQL_DIR / "matriculaciones_insert.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

In [47]:
#PARQUE
lines = []
lines.append("USE automocion;")
lines.append("DROP TABLE IF EXISTS parque;")
lines.append("""
CREATE TABLE parque (
    anio INT,
    camiones_furgonetas INT,
    autobuses INT,
    turismos INT,
    motocicletas INT,
    tractores_industriales INT,
    remolques INT,
    otros_vehiculos INT,
    total INT
);
""")

for _, row in parque.iterrows():
    lines.append(
        f"INSERT INTO parque VALUES "
        f"({int(row['anio'])}, {int(row['camiones_furgonetas'])}, {int(row['autobuses'])}, "
        f"{int(row['turismos'])}, {int(row['motocicletas'])}, "
        f"{int(row['tractores_industriales'])}, {int(row['remolques'])}, "
        f"{int(row['otros_vehiculos'])}, {int(row['total'])});"
    )

with open(SQL_DIR / "parque_insert.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))


In [ ]:
#MATRIULACIONES POR PROVINCIA
lines = []
lines.append("USE automocion;")
lines.append("DROP TABLE IF EXISTS matriculaciones_provincia;")
lines.append("""
CREATE TABLE matriculaciones_provincia (
    provincia VARCHAR(100),
    anio INT,
    matriculaciones INT
);
""")

for _, row in matr_prov.iterrows():
    provincia = str(row["provincia"]).replace("'", "")
    lines.append(
        f"INSERT INTO matriculaciones_provincia VALUES "
        f"('{provincia}', {int(row['anio'])}, {int(row['matriculaciones'])});"
    )

with open(SQL_DIR / "matriculaciones_provincia_insert.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

In [52]:
distintivo.columns = (
    distintivo.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [53]:
#DISTINTIVO AMBIENTAL
lines = []
lines.append("USE automocion;")
lines.append("DROP TABLE IF EXISTS distintivo;")
lines.append("""
CREATE TABLE distintivo (
    provincia VARCHAR(100),
    cero INT,
    b INT,
    c INT,
    eco INT,
    sin_distintivo INT,
    total INT,
    pct_cero FLOAT,
    pct_eco FLOAT,
    pct_c FLOAT,
    pct_b FLOAT,
    pct_sin FLOAT
);
""")

for _, row in distintivo.iterrows():
    provincia = str(row["provincia"]).replace("'", "")
    lines.append(
        f"INSERT INTO distintivo VALUES "
        f"('{provincia}', {int(row['cero'])}, {int(row['b'])}, {int(row['c'])}, "
        f"{int(row['eco'])}, {int(row['sin_distintivo'])}, {int(row['total'])}, "
        f"{row['pct_cero']}, {row['pct_eco']}, {row['pct_c']}, {row['pct_b']}, {row['pct_sin']});"
    )

with open(SQL_DIR / "distintivo_insert.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

In [49]:
#COMBUSTIBLE
lines = []
lines.append("USE automocion;")
lines.append("DROP TABLE IF EXISTS combustible;")
lines.append("""
CREATE TABLE combustible (
    anio INT,
    gasolina INT,
    diesel INT,
    otros INT,
    total INT
);
""")

for _, row in combustible.iterrows():
    lines.append(
        f"INSERT INTO combustible VALUES "
        f"({int(row['anio'])}, {int(row['gasolina'])}, {int(row['diesel'])}, "
        f"{int(row['otros'])}, {int(row['total'])});"
    )

with open(SQL_DIR / "combustible_insert.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

## SQL Business Questions

The following SQL queries were designed to answer key business questions:

1. How has the automotive market evolved over time?
2. What percentage of the vehicle fleet is renewed each year?
3. What is the market share by vehicle type?
4. How has the fuel mix evolved over time?
5. Which provinces account for the largest share of registrations?
6. Which provinces are growing the most?
7. Which provinces have the highest adoption of low-emission vehicles?
8. Which provinces have the oldest vehicle fleet?